# Training v2: improved hyperparameters

Fixes vs `01_train.ipynb`:
- `sigma_rec=0.10` (was 0.05) -- more robust solutions

Saves to `stage{0,1,2}_v2.pt`.

In [ ]:
import sys

sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src import BioLeakyRNN, CuedTargetWithDistractorsV3, TrainConfig, train_supervised

device = "cpu"
print("device:", device)
Path("../checkpoints").mkdir(exist_ok=True)

In [ ]:
def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    ).to(device)

## Stage 0 — detect target, no cue, no distractors

In [ ]:
def make_env_stage0():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=0.0, p_distractor_trial=0.0, distractor_strength=0.0
    )


model = make_model()
cfg0 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
    stop_on_no_miss=0,
)
history0 = train_supervised(model, make_env_stage0, cfg0)
torch.save({"state_dict": model.state_dict()}, "../checkpoints/stage0_v2.pt")
print("Saved: checkpoints/stage0_v2.pt")

## Stage 1 — add cue

In [ ]:
def make_env_stage1():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.0, distractor_strength=0.0
    )


model = make_model()
model.load_state_dict(
    torch.load("../checkpoints/stage0_v2.pt", weights_only=True)["state_dict"]
)
cfg1 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
    stop_on_no_miss=0,
)
history1 = train_supervised(model, make_env_stage1, cfg1)
torch.save({"state_dict": model.state_dict()}, "../checkpoints/stage1_v2.pt")
print("Saved: checkpoints/stage1_v2.pt")

## Stage 2 — cue + distractors (with early stopping)

In [ ]:
def make_env_stage2():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model()
model.load_state_dict(
    torch.load("../checkpoints/stage1_v2.pt", weights_only=True)["state_dict"]
)
cfg2 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=15000,
    print_every=50,
    device=device,
)  # early stopping enabled (default stop_on_no_miss=3)
history2 = train_supervised(model, make_env_stage2, cfg2)
torch.save({"state_dict": model.state_dict()}, "../checkpoints/stage2_v2.pt")
print("Saved: checkpoints/stage2_v2.pt")

## Training curves

In [ ]:
def smooth(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, hist, stage in zip(axes, [history0, history1, history2], [0, 1, 2]):
    updates = np.arange(1, len(hist["p_correct"]) + 1)
    sw = 20
    for key, color, label in [
        ("p_correct", "green", "correct"),
        ("p_miss", "orange", "miss"),
        ("p_abort", "red", "abort"),
    ]:
        vals = hist[key]
        ax.plot(updates, vals, alpha=0.15, color=color, lw=0.8)
        if len(vals) >= sw:
            ax.plot(updates[sw - 1 :], smooth(vals, sw), color=color, lw=2, label=label)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Stage {stage} (v2)")
    ax.set_xlabel("update")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, hist, stage in zip(axes, [history0, history1, history2], [0, 1, 2]):
    updates = np.arange(1, len(hist["loss"]) + 1)
    sw = 20
    ax.plot(updates, hist["loss"], alpha=0.15, color="steelblue", lw=0.8)
    if len(hist["loss"]) >= sw:
        ax.plot(
            updates[sw - 1 :],
            smooth(hist["loss"], sw),
            color="steelblue",
            lw=2,
            label="total",
        )
    ax.plot(updates, hist["ce"], alpha=0.15, color="tomato", lw=0.8)
    if len(hist["ce"]) >= sw:
        ax.plot(
            updates[sw - 1 :], smooth(hist["ce"], sw), color="tomato", lw=2, label="CE"
        )
    ax.set_title(f"Stage {stage} loss (v2)")
    ax.set_xlabel("update")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Quick eval on stage2_v2

In [ ]:
from src.analysis import collect_trials, filter_trials

trials = collect_trials(model, make_env_stage2, n_trials=2000, device=device)
outcomes = {}
for tr in trials:
    o = tr["train_outcome"]
    outcomes[o] = outcomes.get(o, 0) + 1
total = len(trials)
print(f"Total trials: {total}")
for o, n in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")